# 17 — Refine notebook 12/13 features and blend their predictions

**Henrik Remmen (541747), Group 119 — [119] SINTEF Databreach**

The working baseline is the physics XGBoost/LightGBM blend, public score 0.98990.
Notebook 16's new blend scored 0.98643 and changed both features and model capacity.
This experiment holds learning rate, depth/leaves, regularization, seeds and per-unit
tree counts fixed at the successful 12/13 settings. Only the feature set changes.

New family A: fraction of better hours and average price premium before/after each hour,
over 24, 72 and 168 hours. Family B: timing of inflow relative to profitable hours,
inflow-weighted prices, forthcoming reservoir floors and distance to remaining price peaks.
Test A first, then A+B. The full supplied weekly price/inflow paths are legitimate case
inputs. No observed storage after case start, commitment labels, fitted hydraulic simulator,
external data or solver outputs are used in feature engineering.

Run all cells in a fresh kernel. TRAINING A compares original, A and A+B on 2021 for
each model, picks the best NEW variant per backend, and audits the fixed 50/50 pair on 2022.
These historical years were used before, so the audit is not a pristine holdout. Fixed baseline
counts were also chosen in earlier development. TRAINING B always exports the new pair and
its blend, even if the audit is worse; it clearly reports whether improvement is supported.
The last cell exports a short notebook that trains only those two models from raw data.

The first code cell selects local (default 12 CPU threads) or professor (max four) mode.
Fits remain sequential to control memory. Float32 matrices; resumable local checkpoints.
Changing thread count reuses completed local tasks; small floating-point differences across
thread counts are possible. Professor mode disables cache reads for honest cold-run timing.
The exported short notebook ALWAYS starts in professor mode, regardless of the local setting.
Development: 140 unit fits. Final reproduction: 28 fits. Allow hours for development;
runtime and peak RAM during actual fitting are not yet measured. The professor's final-only
notebook has no feature search or validation runs and must be cold-timed against 12 hours/32 GB.
All required code is embedded. Libraries are listed in the project requirements.txt.
No model training was performed when this notebook was constructed. No score is guaranteed.

In [1]:
# RESOURCE PROFILE — run this cell first, in a fresh kernel.
import os

RUN_PROFILE = 'local'  # 'local' for this Mac; 'professor' for a four-core cold run
LOCAL_CPU_THREADS = 12  # Change to 8 if the Mac becomes sluggish; speed is not benchmarked.

if RUN_PROFILE not in ('local', 'professor'):
    raise ValueError("RUN_PROFILE must be 'local' or 'professor'")
THREADS = min(LOCAL_CPU_THREADS if RUN_PROFILE == 'local' else 4, os.cpu_count() or 1)
if THREADS < 1:
    raise ValueError('CPU thread count must be positive')
# Professor timing must include actual fitting, not restored predictions.
USE_CHECKPOINTS = RUN_PROFILE == 'local'
for key in ('OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[key] = '1'
os.environ['OMP_NUM_THREADS'] = str(THREADS)
print(f'Profile: {RUN_PROFILE} | {THREADS} threads per model | one model at a time')
print('Restore completed tasks:', USE_CHECKPOINTS)

Profile: local | 12 threads per model | one model at a time
Restore completed tasks: True


In [2]:
from pathlib import Path
import os
import time
from IPython.display import display

NOTEBOOK_STARTED = time.perf_counter()
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'data/kernel').is_dir() and (p / 'prediction_mapping.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Place notebook beside the supplied data/ directory and mapping files.')
SOURCE_DIGEST = '94b35b1ac9206f02683466ec302e8ee1e1ac66ea92847b5ac549639a63071e7b'

# Existing models are frozen controls. Their counts were determined in prior development.
BASELINE_ROUNDS = {
 'xgb_v2': dict(zip(
  ['Hogga_G1','Tokke_G1','Tokke_G2','Tokke_G3','Tokke_G4','Lio_G1','Byrte_G1',
   'Vinje_G1','Vinje_G2','Vinje_G3','Haukeli_G1','Songa_G1','Kjela_G1','Vesle Kjela_G1'],
  [410,450,854,587,693,249,292,658,404,607,223,510,326,372])),
 'lgb_v2': dict(zip(
  ['Hogga_G1','Tokke_G1','Tokke_G2','Tokke_G3','Tokke_G4','Lio_G1','Byrte_G1',
   'Vinje_G1','Vinje_G2','Vinje_G3','Haukeli_G1','Songa_G1','Kjela_G1','Vesle Kjela_G1'],
  [398,534,387,458,410,386,441,852,299,873,330,465,393,177]))}
xgb_parameters = dict(objective='binary:logistic', eval_metric='auc', learning_rate=.03,
    max_depth=7, min_child_weight=50, subsample=.85, colsample_bytree=.85,
    reg_alpha=.1, reg_lambda=5., tree_method='hist', max_bin=256, device='cpu', random_state=42)
lgb_parameters = dict(objective='binary', metric='None', learning_rate=.03, num_leaves=31,
    max_depth=-1, min_data_in_leaf=150, feature_fraction=.85, bagging_fraction=.85,
    bagging_freq=1, lambda_l1=.1, lambda_l2=5., max_bin=255, device_type='cpu',
    deterministic=True, force_col_wise=True, seed=42, verbosity=-1)
SPECS = {}
for family, backend, parameters, cap in [
    ('xgb', 'xgboost', xgb_parameters, 2000),
    ('lgb', 'lightgbm', lgb_parameters, 1800),
]:
    for variant in ('physics', 'competition', 'competition_timing'):
        name = family + ('_v2' if variant=='physics' else '_' + variant)
        SPECS[name] = dict(backend=backend, features=variant, max_rounds=cap, parameters=dict(parameters))
print(f'CPU threads: {THREADS}. Sequential fits; notebook 12/13 settings and rounds are frozen.')

CPU threads: 12. Sequential fits; notebook 12/13 settings and rounds are frozen.


## pipeline — definitions only

In [3]:
"""Shared, target-safe data and feature pipeline used by experiment notebooks.

The module contains no model training.  It exists so CatBoost and XGBoost
experiments use exactly the same inputs, folds, metric, and submission layout.
"""

from dataclasses import dataclass
from pathlib import Path
import re

import numpy as np
import pandas as pd
import yaml
from sklearn.metrics import roc_auc_score


HORIZON = 168
RANDOM_SEED = 42


@dataclass
class ProjectData:
    root: Path
    X_all: pd.DataFrame
    target_cube: np.ndarray
    units: list[str]
    train_cases: pd.DataFrame
    test_cases: pd.DataFrame
    sample_submission: pd.DataFrame
    feature_sets: dict[str, list[str]]
    feature_group: dict[str, str]
    generator_features: pd.DataFrame
    folds: list[dict]

    @property
    def n_train_cases(self):
        return len(self.train_cases)

    @property
    def n_test_cases(self):
        return len(self.test_cases)

    @property
    def n_all_cases(self):
        return self.n_train_cases + self.n_test_cases


def read_time_csv(path, time_column="date"):
    frame = pd.read_csv(path)
    frame[time_column] = pd.to_datetime(frame[time_column], utc=True)
    return frame


def rows_for_cases(case_ids):
    case_ids = np.asarray(case_ids)
    return (
        case_ids[:, None] * HORIZON
        + np.arange(HORIZON)[None, :]
    ).ravel()


def micro_auc(y_true_cube, y_pred_cube):
    if y_true_cube.shape != y_pred_cube.shape:
        raise ValueError("Truth and prediction cubes must have identical shapes")
    return roc_auc_score(
        y_true_cube.reshape(len(y_true_cube), -1),
        y_pred_cube.reshape(len(y_pred_cube), -1),
        average="micro",
    )


def per_unit_auc(y_true_cube, y_pred_cube, units):
    return {
        unit: roc_auc_score(
            y_true_cube[:, unit_index, :].ravel(),
            y_pred_cube[:, unit_index, :].ravel(),
        )
        for unit_index, unit in enumerate(units)
    }


def make_purged_folds(train_cases, validation_years=(2020, 2021, 2022)):
    starts = pd.DatetimeIndex(train_cases["starttime"])
    folds = []
    for year in validation_years:
        validation_start = pd.Timestamp(f"{year}-01-01", tz="UTC")
        validation_end = pd.Timestamp(f"{year + 1}-01-01", tz="UTC")
        validation_cases = np.flatnonzero(
            (starts >= validation_start) & (starts < validation_end)
        )
        training_cases = np.flatnonzero(
            starts + pd.Timedelta(hours=HORIZON - 1) < validation_start
        )
        if not len(training_cases) or not len(validation_cases):
            raise ValueError(f"Empty training or validation partition for {year}")
        if not (
            starts[training_cases].max() + pd.Timedelta(hours=HORIZON - 1)
            < starts[validation_cases].min()
        ):
            raise AssertionError(f"Fold {year} is not temporally purged")
        folds.append({
            "year": year,
            "train_cases": training_cases,
            "validation_cases": validation_cases,
        })
    return folds


def make_submission(data, prediction_cube, filename):
    expected = (data.n_test_cases, len(data.units), HORIZON)
    if prediction_cube.shape != expected:
        raise ValueError(f"Expected predictions {expected}, got {prediction_cube.shape}")

    submission_data = {"Run No": data.test_cases["Run No"].to_numpy()}
    for unit_index, unit in enumerate(data.units):
        for horizon in range(HORIZON):
            submission_data[f"result_committed_{unit}_t{horizon}"] = (
                prediction_cube[:, unit_index, horizon]
            )

    submission = pd.DataFrame(submission_data)[data.sample_submission.columns]
    probabilities = submission.iloc[:, 1:].to_numpy()
    if not submission["Run No"].equals(data.sample_submission["Run No"]):
        raise AssertionError("Run No does not match sample_submission.csv")
    if not np.isfinite(probabilities).all():
        raise ValueError("Submission contains missing or non-finite predictions")
    if probabilities.min() < 0 or probabilities.max() > 1:
        raise ValueError("Submission probabilities are outside [0, 1]")

    path = data.root / "submissions" / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    submission.to_csv(path, index=False)
    return path, submission


def build_project_data(root):
    root = Path(root).resolve()
    data_dir = root / "data"

    price = read_time_csv(data_dir / "kernel" / "Historical_day_ahead_price_2015_2025.csv")
    inflow = read_time_csv(data_dir / "kernel" / "Historical_inflow_1958_2025.csv")
    volume = read_time_csv(data_dir / "kernel" / "Historical_volume_2015_2024.csv")
    water_value = read_time_csv(data_dir / "kernel" / "Synthetic_water_value_2015_2024.csv")
    min_flow = read_time_csv(data_dir / "extended" / "Constraint_min_flow.csv")
    min_volume = read_time_csv(data_dir / "extended" / "Constraint_min_volume.csv")
    targets = read_time_csv(data_dir / "kernel" / "Unit_commitment_decisions.csv", "starttime")
    prediction_mapping = read_time_csv(root / "prediction_mapping.csv", "starttime")
    sample_submission = pd.read_csv(root / "sample_submission.csv")

    with open(data_dir / "extended" / "Tokke_Vinje_topology.yaml") as stream:
        topology = yaml.safe_load(stream)

    pattern = re.compile(r"^result_committed_(.+)_t(\d+)$")
    target_columns = [column for column in targets if pattern.match(column)]
    units = list(dict.fromkeys(pattern.match(column).group(1) for column in target_columns))
    if len(units) != 14 or len(target_columns) != 14 * HORIZON:
        raise AssertionError("Expected 14 units and 168 targets per unit")
    if target_columns != list(sample_submission.columns[1:]):
        raise AssertionError("Target order differs from sample submission")

    target_cube = np.stack([
        targets[[f"result_committed_{unit}_t{hour}" for hour in range(HORIZON)]]
        .to_numpy(dtype=np.int8)
        for unit in units
    ], axis=1)
    if not set(np.unique(target_cube)).issubset({0, 1}):
        raise AssertionError("Targets must be binary")

    train_cases = targets[["Run No", "starttime"]].copy()
    test_cases = prediction_mapping[["Run No", "starttime"]].copy()
    all_cases = pd.concat([train_cases, test_cases], ignore_index=True)
    n_all_cases = len(all_cases)
    starts = pd.DatetimeIndex(all_cases["starttime"])
    horizon_hour = np.tile(np.arange(HORIZON), n_all_cases)
    target_time = pd.DatetimeIndex(
        starts.repeat(HORIZON) + pd.to_timedelta(horizon_hour, unit="h")
    )

    def numeric_indexed(frame):
        return frame.set_index("date").select_dtypes(include=np.number)

    def hourly_matrix(series):
        values = series.reindex(target_time).to_numpy()
        matrix = values.reshape(n_all_cases, HORIZON)
        if not np.isfinite(matrix).all():
            raise ValueError(
                f"Missing/non-finite hourly inputs: {(~np.isfinite(matrix)).sum()}"
            )
        return matrix

    def repeat_case(values):
        values = np.asarray(values)
        if len(values) != n_all_cases:
            raise ValueError("Case-level array has incorrect length")
        return np.repeat(values, HORIZON)

    def cyclic(values, period):
        angle = 2 * np.pi * np.asarray(values) / period
        return np.sin(angle), np.cos(angle)

    def clipped_shift(matrix, offset):
        indices = np.clip(np.arange(HORIZON) + offset, 0, HORIZON - 1)
        return matrix[:, indices]

    def centred_rolling_mean(matrix, window):
        return (
            pd.DataFrame(matrix).T
            .rolling(window=window, min_periods=1, center=True)
            .mean().T.to_numpy()
        )

    feature_data = {}
    feature_group = {}

    def add_feature(name, values, group):
        values = np.asarray(values)
        expected = n_all_cases * HORIZON
        if values.ndim != 1 or len(values) != expected:
            raise ValueError(f"{name}: expected {expected}, got {values.shape}")
        feature_data[name] = values.astype(np.float32)
        feature_group[name] = group

    hour = target_time.hour.to_numpy()
    weekday = target_time.dayofweek.to_numpy()
    month = target_time.month.to_numpy()
    for name, values, period in (
        ("hour", hour, 24),
        ("weekday", weekday, 7),
        ("month", month - 1, 12),
    ):
        sine, cosine = cyclic(values, period)
        add_feature(f"{name}_sin", sine, "calendar")
        add_feature(f"{name}_cos", cosine, "calendar")
    add_feature("horizon_hour", horizon_hour, "calendar")

    price_matrix = hourly_matrix(
        price.set_index("date")["Day-ahead price (EUR/MWh)"]
    )
    price_mean = price_matrix.mean(axis=1)
    price_std = price_matrix.std(axis=1)
    price_rank = pd.DataFrame(price_matrix).rank(axis=1, pct=True).to_numpy()
    add_feature("price", price_matrix.ravel(), "price")
    add_feature("price_week_mean", repeat_case(price_mean), "price")
    add_feature("price_week_std", repeat_case(price_std), "price")
    add_feature("price_week_percentile", price_rank.ravel(), "price")
    add_feature("price_minus_week_mean", (price_matrix - price_mean[:, None]).ravel(), "price")
    add_feature(
        "price_week_zscore",
        ((price_matrix - price_mean[:, None]) / (price_std[:, None] + 1e-6)).ravel(),
        "price_windows",
    )
    add_feature("price_top_10pct", (price_rank >= 0.90).ravel(), "price_windows")
    add_feature("price_bottom_10pct", (price_rank <= 0.10).ravel(), "price_windows")
    for offset in (1, 3, 6, 24):
        lag = clipped_shift(price_matrix, -offset)
        lead = clipped_shift(price_matrix, offset)
        add_feature(f"price_lag_{offset}h", lag.ravel(), "price_windows")
        add_feature(f"price_lead_{offset}h", lead.ravel(), "price_windows")
        add_feature(f"price_change_from_{offset}h", (price_matrix - lag).ravel(), "price_windows")
        add_feature(f"price_change_to_next_{offset}h", (lead - price_matrix).ravel(), "price_windows")
    for window in (3, 6, 12, 24, 48):
        rolling = centred_rolling_mean(price_matrix, window)
        add_feature(f"price_mean_{window}h", rolling.ravel(), "price_windows")
        add_feature(f"price_minus_mean_{window}h", (price_matrix - rolling).ravel(), "price_windows")

    reservoirs = [column for column in volume.columns if column != "date"]
    capacity = {
        name: values["max_vol"]
        for name, values in topology["model"]["reservoir"].items()
    }
    start_volume = numeric_indexed(volume).reindex(starts)[reservoirs]
    start_water = numeric_indexed(water_value).reindex(starts)[reservoirs]
    end_water = numeric_indexed(water_value).reindex(
        starts + pd.Timedelta(days=7)
    )[reservoirs]
    if start_volume.isna().any().any() or start_water.isna().any().any() or end_water.isna().any().any():
        raise ValueError("Missing case-level reservoir data")

    for reservoir in reservoirs:
        fill = start_volume[reservoir].to_numpy() / capacity[reservoir]
        water_start = start_water[reservoir].to_numpy()
        water_end = end_water[reservoir].to_numpy()
        add_feature(f"fill__{reservoir}", repeat_case(fill), "hydrology")
        add_feature(f"water_value_end__{reservoir}", repeat_case(water_end), "water_margins")
        add_feature(f"water_value_change__{reservoir}", repeat_case(water_end - water_start), "water_margins")
        add_feature(f"price_margin__{reservoir}", (price_matrix - water_end[:, None]).ravel(), "water_margins")

    system_fill = (
        start_volume.sum(axis=1).to_numpy()
        / sum(capacity[reservoir] for reservoir in reservoirs)
    )
    water_end_mean = end_water.mean(axis=1).to_numpy()
    add_feature("system_filling_ratio", repeat_case(system_fill), "hydrology")
    add_feature("water_value_end_mean", repeat_case(water_end_mean), "water_margins")
    margin = price_matrix - water_end_mean[:, None]
    add_feature("price_minus_water_value_mean", margin.ravel(), "water_margins")
    add_feature(
        "price_to_abs_water_value_ratio",
        (price_matrix / (np.abs(water_end_mean[:, None]) + 1.0)).ravel(),
        "water_margins",
    )

    inflow_values = numeric_indexed(inflow)
    inflow_matrices = {}
    for column in inflow_values:
        matrix = hourly_matrix(inflow_values[column])
        inflow_matrices[column] = matrix
        add_feature(f"inflow__{column}", matrix.ravel(), "hydrology")
        add_feature(f"inflow_week_mean__{column}", repeat_case(matrix.mean(axis=1)), "hydrology")
    total_inflow = np.sum(list(inflow_matrices.values()), axis=0)
    add_feature("inflow_total", total_inflow.ravel(), "hydrology")
    add_feature("inflow_total_week_mean", repeat_case(total_inflow.mean(axis=1)), "hydrology")
    add_feature("inflow_total_week_std", repeat_case(total_inflow.std(axis=1)), "hydrology")
    add_feature("inflow_total_24h_mean", centred_rolling_mean(total_inflow, 24).ravel(), "hydrology")
    add_feature(
        "inflow_total_cumulative_mean",
        (np.cumsum(total_inflow, axis=1) / (np.arange(HORIZON)[None, :] + 1)).ravel(),
        "hydrology",
    )

    flow_values = numeric_indexed(min_flow)
    min_volume_values = numeric_indexed(min_volume)
    flow_matrices = {}
    min_volume_matrices = {}
    for column in flow_values:
        matrix = hourly_matrix(flow_values[column])
        flow_matrices[column] = matrix
        add_feature(f"min_flow__{column}", matrix.ravel(), "constraints")
    for column in min_volume_values:
        matrix = hourly_matrix(min_volume_values[column])
        min_volume_matrices[column] = matrix
        add_feature(f"min_volume__{column}", matrix.ravel(), "constraints")
        if column in reservoirs:
            initial = start_volume[column].to_numpy()[:, None]
            add_feature(
                f"min_volume_slack__{column}",
                ((initial - matrix) / capacity[column]).ravel(),
                "constraints",
            )

    flow_active_count = np.sum([matrix > 0 for matrix in flow_matrices.values()], axis=0)
    volume_active_count = np.sum([matrix > 0 for matrix in min_volume_matrices.values()], axis=0)
    total_min_flow = np.sum(list(flow_matrices.values()), axis=0)
    total_min_volume = np.sum(list(min_volume_matrices.values()), axis=0)
    add_feature("min_flow_active_count", flow_active_count.ravel(), "constraints")
    add_feature("min_volume_active_count", volume_active_count.ravel(), "constraints")
    add_feature("min_flow_total", total_min_flow.ravel(), "constraints")
    add_feature("min_volume_total", total_min_volume.ravel(), "constraints")

    variable_flow_columns = [
        column for column in flow_values
        if 0 < (flow_values[column] > 0).mean() < 1
    ]
    variable_volume_columns = [
        column for column in min_volume_values
        if 0 < (min_volume_values[column] > 0).mean() < 1
    ]
    variable_active = np.zeros((n_all_cases, HORIZON), dtype=bool)
    for column in variable_flow_columns:
        variable_active |= flow_matrices[column] > 0
    for column in variable_volume_columns:
        variable_active |= min_volume_matrices[column] > 0
    add_feature("variable_constraint_active", variable_active.ravel(), "constraints")

    add_feature("margin_x_system_fill", (margin * system_fill[:, None]).ravel(), "interactions")
    add_feature("price_rank_x_system_fill", (price_rank * system_fill[:, None]).ravel(), "interactions")
    add_feature("margin_x_total_inflow", (margin * total_inflow).ravel(), "interactions")
    add_feature("price_rank_x_hour_sin", price_rank.ravel() * feature_data["hour_sin"], "interactions")
    add_feature("price_rank_x_hour_cos", price_rank.ravel() * feature_data["hour_cos"], "interactions")
    add_feature("margin_x_constraint", margin.ravel() * variable_active.ravel(), "interactions")

    X_all = pd.DataFrame(feature_data)
    constants = [
        column for column in X_all
        if X_all[column].nunique(dropna=False) <= 1
    ]
    X_all = X_all.drop(columns=constants)
    if X_all.isna().any().any() or not np.isfinite(X_all.to_numpy()).all():
        raise ValueError("Final feature matrix contains invalid values")

    baseline = [
        "hour_sin", "hour_cos", "weekday_sin", "weekday_cos",
        "month_sin", "month_cos", "horizon_hour", "price",
        "price_week_mean", "price_week_std", "price_week_percentile",
        "price_minus_week_mean", "water_value_end_mean",
        "price_minus_water_value_mean", "system_filling_ratio",
        "inflow_total", "inflow_total_24h_mean", "min_flow_total",
        "min_volume_total", "variable_constraint_active",
    ]

    def grouped(name):
        return [
            feature for feature, group in feature_group.items()
            if group == name and feature in X_all.columns
        ]

    def unique_existing(names):
        return list(dict.fromkeys(name for name in names if name in X_all.columns))

    feature_sets = {
        "baseline": unique_existing(baseline),
        "baseline_plus_price": unique_existing(baseline + grouped("price_windows")),
        "baseline_plus_price_water": unique_existing(
            baseline + grouped("price_windows") + grouped("water_margins")
        ),
        "baseline_plus_price_water_hydrology": unique_existing(
            baseline + grouped("price_windows") + grouped("water_margins")
            + grouped("hydrology")
        ),
        "full_engineered": unique_existing(
            baseline + grouped("price_windows") + grouped("water_margins")
            + grouped("hydrology") + grouped("constraints")
            + grouped("interactions")
        ),
    }

    generator_rows = []
    generator_config = topology["model"]["generator"]
    for unit in units:
        config = generator_config[unit]
        plant = unit.rsplit("_G", 1)[0]
        number_match = re.search(r"_G(\d+)$", unit)
        generator_rows.append({
            "unit": unit,
            "plant": plant,
            "generator_number": int(number_match.group(1)) if number_match else 1,
            "p_min": config.get("p_min", np.nan),
            "p_max": config.get("p_max", np.nan),
            "p_nom": config.get("p_nom", np.nan),
            "minimum_output_fraction": config.get("p_min", np.nan) / config.get("p_max", np.nan),
            "start_cost": config.get("startcost_const", 0.0),
            "stop_cost": config.get("stopcost_const", 0.0),
            "penstock": str(config.get("penstock", "unknown")),
        })
    generator_features = pd.DataFrame(generator_rows)
    generator_features["units_in_plant"] = (
        generator_features.groupby("plant")["unit"].transform("count")
    )

    return ProjectData(
        root=root,
        X_all=X_all,
        target_cube=target_cube,
        units=units,
        train_cases=train_cases,
        test_cases=test_cases,
        sample_submission=sample_submission,
        feature_sets=feature_sets,
        feature_group=feature_group,
        generator_features=generator_features,
        folds=make_purged_folds(train_cases),
    )

## physics — definitions only

In [4]:
"""Deterministic input-only features for the next hydropower experiments.

No fitted transforms, labels, predicted/observed future storage, or solver outputs.
Water budgets deliberately exclude unknown dispatch and are explicitly proxies.
"""

import numpy as np
import pandas as pd
import yaml



def window_sum(values, width, forward=True):
    """Truncated within-case sum, including the current hour; no edge repeats."""
    if not forward:
        return window_sum(values[:, ::-1], width, True)[:, ::-1]
    cumulative = np.pad(np.cumsum(values, axis=1, dtype=np.float64), ((0, 0), (1, 0)))
    left = np.arange(values.shape[1])
    right = np.minimum(left + width, values.shape[1])
    return cumulative[:, right] - cumulative[:, left]


def consecutive_true(mask):
    """Run length ending/starting at each position, reset at each case boundary."""
    past = np.zeros(mask.shape, dtype=np.float32)
    future = np.zeros_like(past)
    for h in range(mask.shape[1]):
        past[:, h] = mask[:, h] * (1 + (past[:, h - 1] if h else 0))
    for h in range(mask.shape[1] - 1, -1, -1):
        future[:, h] = mask[:, h] * (1 + (future[:, h + 1] if h + 1 < mask.shape[1] else 0))
    return past, future


def plant_sources(topology, plant):
    """Follow reservoir -> plant or reservoir -> tunnel -> plant connections.

    Stop at the first reservoir; do not pretend all upstream water arrives instantly.
    """
    connections = topology['connections']
    found = set()

    def visit(name, kind, seen):
        if (name, kind) in seen:
            return
        seen = seen | {(name, kind)}
        for edge in connections:
            if edge['to'] != name or edge['to_type'] != kind:
                continue
            if edge['from_type'] == 'reservoir':
                found.add(edge['from'])
            elif edge['from_type'] == 'tunnel':
                visit(edge['from'], 'tunnel', seen)

    visit(plant, 'plant', set())
    if not found:
        raise ValueError(f'No source reservoir found for {plant}')
    return sorted(found)


def build_research_features(data):
    """Return global additions, per-plant additions, and an auditable source mapping."""
    root = data.root
    with open(root / 'data/extended/Tokke_Vinje_topology.yaml') as stream:
        topology = yaml.safe_load(stream)
    starts = pd.DatetimeIndex(pd.concat([data.train_cases, data.test_cases])['starttime'])
    n = data.n_all_cases
    horizon = np.arange(HORIZON)
    time_index = starts.repeat(HORIZON) + pd.to_timedelta(np.tile(horizon, n), unit='h')
    price = data.X_all['price'].to_numpy().reshape(n, HORIZON)
    volume = read_time_csv(root / 'data/kernel/Historical_volume_2015_2024.csv').set_index('date').reindex(starts)
    water = read_time_csv(root / 'data/kernel/Synthetic_water_value_2015_2024.csv').set_index('date').reindex(starts + pd.Timedelta(days=7))
    inflow = read_time_csv(root / 'data/kernel/Historical_inflow_1958_2025.csv').set_index('date').reindex(time_index)
    minimum = read_time_csv(root / 'data/extended/Constraint_min_volume.csv').set_index('date').reindex(time_index)
    extra = {}

    def add(target, name, values):
        values = np.broadcast_to(values, (n, HORIZON)).reshape(-1).astype(np.float32)
        if not np.isfinite(values).all():
            raise ValueError(f'Non-finite research feature {name}')
        target[name] = values

    add(extra, 'v2_day_in_horizon', np.broadcast_to(horizon // 24, (n, HORIZON)))
    add(extra, 'v2_hours_remaining', HORIZON - horizon)
    day_price = price.reshape(n, 7, 24)
    day_mean = np.repeat(day_price.mean(axis=2), 24, axis=1)
    day_rank = pd.DataFrame(day_price.reshape(-1, 24)).rank(axis=1, pct=True).to_numpy().reshape(n, HORIZON)
    add(extra, 'v2_price_day_rank', day_rank)
    add(extra, 'v2_price_minus_day_mean', price - day_mean)
    prefix_mean = np.cumsum(price, axis=1) / (horizon + 1)
    suffix_mean = np.cumsum(price[:, ::-1], axis=1)[:, ::-1] / (HORIZON - horizon)
    add(extra, 'v2_price_minus_prefix_mean', price - prefix_mean)
    add(extra, 'v2_price_minus_suffix_mean', price - suffix_mean)
    add(extra, 'v2_best_remaining_price_gap', np.maximum.accumulate(price[:, ::-1], axis=1)[:, ::-1] - price)
    reservoirs = topology['model']['reservoir']
    reservoir_inputs = {}
    for name, config in reservoirs.items():
        cap = float(config['max_vol'])
        initial = volume[name].to_numpy()[:, None]
        terminal = water[name].to_numpy()[:, None]
        flow = inflow[name].to_numpy().reshape(n, HORIZON)
        # m3/s * 3600 / 1e6 -> million m3 per hour, matching storage units.
        hourly_volume = flow * 0.0036
        cumulative = np.cumsum(hourly_volume, axis=1)
        remaining = np.cumsum(hourly_volume[:, ::-1], axis=1)[:, ::-1]
        floor = minimum[name].to_numpy().reshape(n, HORIZON) if name in minimum else np.zeros_like(price)
        budget = initial + cumulative
        add(extra, f'v2_cumulative_inflow_fill__{name}', cumulative / cap)
        add(extra, f'v2_remaining_inflow_fill__{name}', remaining / cap)
        add(extra, f'v2_no_release_excess__{name}', np.maximum(budget - cap, 0) / cap)
        add(extra, f'v2_no_release_min_slack__{name}', (budget - floor) / cap)
        curve = config['vol_head']
        head = np.interp(initial[:, 0], curve['x'], curve['y'])[:, None]
        reservoir_inputs[name] = (cap, initial, terminal, cumulative, remaining, head)

    plants = {}
    mappings = []
    for plant, plant_config in topology['model']['plant'].items():
        sources = plant_sources(topology, plant)
        values = [reservoir_inputs[s] for s in sources]
        capacity = sum(v[0] for v in values)
        initial = sum(v[1] for v in values)
        # Multi-reservoir summaries are capacity-weighted approximations, not SHOP prices.
        terminal = sum(v[2] * v[0] for v in values) / capacity
        head = sum(v[5] * v[0] for v in values) / capacity - float(plant_config['outlet_line'])
        cum = sum(v[3] for v in values)
        remain = sum(v[4] for v in values)
        margin = price - terminal
        out = {}
        add(out, 'local_initial_fill', initial / capacity)
        add(out, 'local_gross_head_proxy', head)
        add(out, 'local_terminal_water_value', terminal)
        add(out, 'local_margin', margin)
        add(out, 'local_margin_normalized', margin / (np.abs(terminal) + price.std(axis=1)[:, None] + 1))
        add(out, 'local_cumulative_inflow_fill', cum / capacity)
        add(out, 'local_remaining_inflow_fill', remain / capacity)
        add(out, 'local_margin_x_fill', margin * initial / capacity)
        positive = np.maximum(margin, 0)
        negative = np.maximum(-margin, 0)
        past, future = consecutive_true(margin > 0)
        add(out, 'local_profitable_run_past', past)
        add(out, 'local_profitable_run_future', future)
        add(out, 'local_profitable_fraction_week', (margin > 0).mean(axis=1)[:, None])
        for hours in (6, 24):
            add(out, f'local_positive_margin_next_{hours}h', window_sum(positive, hours))
            add(out, f'local_negative_margin_next_{hours}h', window_sum(negative, hours))
            add(out, f'local_positive_margin_past_{hours}h', window_sum(positive, hours, False))
        # Start cost / rated production gives EUR/MWh; margin sums are hourly
        # EUR/MWh contributions. This is an approximate rated-power payback ratio.
        unit_configs = [topology['model']['generator'][u] for u in data.units if u.rsplit('_G', 1)[0] == plant]
        cost_per_mw = np.mean([(v.get('startcost_const', 0) + v.get('stopcost_const', 0)) / v['p_max'] for v in unit_configs])
        add(out, 'local_start_stop_payback_proxy', window_sum(positive, 24) / max(cost_per_mw, 1e-6))
        plants[plant] = pd.DataFrame(out)
        mappings.append({'plant': plant, 'source_reservoirs': '|'.join(sources), 'source_capacity_Mm3': capacity})
    return pd.DataFrame(extra), plants, pd.DataFrame(mappings)

## runner — definitions only

In [5]:
"""Sequential, resumable challenger. Importing this module never trains models."""
import gc
import hashlib
import importlib.metadata
import json
import os
import time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from threadpoolctl import threadpool_limits


def raw_digest(root):
    paths = sorted((root / 'data').rglob('*.csv')) + sorted((root / 'data').rglob('*.yaml'))
    paths += [root / 'prediction_mapping.csv', root / 'sample_submission.csv']
    digest = hashlib.sha256()
    for path in paths:
        digest.update(str(path.relative_to(root)).encode())
        with path.open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(block)
    return digest.hexdigest()


def save_json(path, value):
    path = Path(path)
    temporary = path.with_suffix('.tmp.json')
    temporary.write_text(json.dumps(value, indent=2), encoding='utf-8')
    os.replace(temporary, path)


def purged_year(data, year):
    starts = pd.DatetimeIndex(data.train_cases['starttime'])
    begin = pd.Timestamp(f'{year}-01-01', tz='UTC')
    end = pd.Timestamp(f'{year+1}-01-01', tz='UTC')
    train = np.flatnonzero(starts + pd.Timedelta(hours=HORIZON-1) < begin)
    valid = np.flatnonzero((starts >= begin) & (starts < end))
    if not len(train) or not len(valid):
        raise ValueError('Empty temporal split')
    assert starts[train].max() + pd.Timedelta(hours=HORIZON-1) < starts[valid].min()
    return train, valid


def score_cube(truth, prediction):
    return float(roc_auc_score(truth.ravel(), prediction.ravel()))


def fit_member(spec, X, y, rounds, threads, valid=None):
    parameters = dict(spec['parameters'])
    if spec['backend'] == 'xgboost':
        from xgboost import XGBClassifier
        model = XGBClassifier(**parameters, n_estimators=int(rounds), n_jobs=threads,
                              **({'early_stopping_rounds': 80} if valid is not None else {}))
        model.fit(X, y, eval_set=[valid] if valid is not None else None, verbose=False)
        best = int(model.best_iteration) + 1 if valid is not None else int(rounds)
    else:
        import lightgbm as lgb
        parameters['num_threads'] = threads
        training = lgb.Dataset(X, label=y, free_raw_data=True)
        options = {}
        if valid is not None:
            parameters['metric'] = 'auc'
            options = dict(valid_sets=[lgb.Dataset(valid[0], label=valid[1], reference=training)],
                           callbacks=[lgb.early_stopping(80, verbose=False)])
        model = lgb.train(parameters, training, num_boost_round=int(rounds), **options)
        best = int(model.best_iteration or rounds)
    return model, best


class ChallengerRunner:
    def __init__(self, data, extra, local, additions, specs, threads, source_digest):
        self.data, self.local, self.additions = data, local, additions
        self.specs, self.threads = specs, max(1, min(int(threads), os.cpu_count() or 1))
        self.columns = list(data.feature_sets['full_engineered']) + list(extra.columns)
        self.base = np.column_stack([data.X_all[data.feature_sets['full_engineered']].to_numpy(np.float32),
                                     extra.to_numpy(np.float32)])
        self.raw = raw_digest(data.root)
        self.source_digest = source_digest
        self.directory = data.root / 'artifacts/physics_features_v4'
        self.directory.mkdir(parents=True, exist_ok=True)

    def run_stage(self, member, train, predict, stage, rounds=None, tune=False):
        spec = self.specs[member]
        manifest = dict(member=member, spec=spec, train=list(map(int, train)),
                        predict=list(map(int, predict)), stage=stage, rounds=rounds,
                        tune=tune, source=self.source_digest, raw=self.raw, threads=self.threads,
                        packages={p: importlib.metadata.version(p) for p in ('numpy', 'pandas', 'scikit-learn', spec['backend'])})
        signature = hashlib.sha256(json.dumps(dict(manifest, threads=4), sort_keys=True).encode()).hexdigest()[:20]
        directory = self.directory / member / signature
        directory.mkdir(parents=True, exist_ok=True)
        save_json(directory / 'manifest.json', manifest)
        cube = np.empty((len(predict), len(self.data.units), HORIZON), np.float32)
        selected, timings = {}, []
        train_rows, pred_rows = rows_for_cases(train), rows_for_cases(predict)
        for i, unit in enumerate(self.data.units):
            start = time.perf_counter()
            checkpoint = directory / f'unit_{i:02d}.npz'
            if USE_CHECKPOINTS and checkpoint.exists():
                with np.load(checkpoint, allow_pickle=False) as saved:
                    values = saved['prediction'].copy()
                    best = int(saved['rounds'])
                status = 'restored'
            else:
                plant = unit.rsplit('_G', 1)[0]
                parts = [self.base, self.local[plant].to_numpy(np.float32)]
                names = self.columns + list(self.local[plant].columns)
                if spec['features'] != 'physics':
                    parts.append(self.additions[spec['features']][plant].to_numpy(np.float32))
                    names += list(self.additions[spec['features']][plant].columns)
                X = np.ascontiguousarray(np.column_stack(parts), dtype=np.float32)
                del parts
                y = self.data.target_cube[train, i].ravel()
                count = int(rounds[unit]) if rounds is not None else int(spec['max_rounds'])
                print(f'{stage} | {member} | {unit} | up to {count} rounds', flush=True)
                if len(np.unique(y)) == 1:
                    values = np.full((len(predict), HORIZON), y[0], np.float32)
                    best = 1
                else:
                    validation = (X[pred_rows], self.data.target_cube[predict, i].ravel()) if tune else None
                    # Limit BLAS as well as each model's own OpenMP threads.
                    with threadpool_limits(limits=self.threads), threadpool_limits(limits=1, user_api='blas'):
                        model, best = fit_member(spec, X[train_rows], y, count, self.threads, validation)
                        if spec['backend'] == 'xgboost':
                            values = model.predict_proba(X[pred_rows])[:, 1]
                            gain = model.feature_importances_
                        else:
                            values = model.predict(X[pred_rows], num_threads=self.threads)
                            gain = model.feature_importance(importance_type='gain')
                        values = values.reshape(len(predict), HORIZON).astype(np.float32)
                    pd.DataFrame({'feature': names, 'gain': gain}).sort_values('gain', ascending=False).to_csv(
                        directory / f'importance_{i:02d}.csv', index=False)
                    del model, validation
                temporary = checkpoint.with_suffix('.tmp.npz')
                np.savez_compressed(temporary, prediction=values, rounds=best)
                os.replace(temporary, checkpoint)
                del X, y
                gc.collect()
                status = 'trained'
            if values.shape != (len(predict), HORIZON) or not np.isfinite(values).all() or values.min() < 0 or values.max() > 1:
                raise ValueError(f'Invalid predictions: {checkpoint}')
            cube[:, i] = values
            selected[unit] = int(best)
            seconds = time.perf_counter() - start
            timings.append(dict(unit=unit, rounds=best, seconds=seconds, status=status))
            pd.DataFrame(timings).to_csv(directory / 'runtime.csv', index=False)
            print(f'  {status}: {best} rounds, {seconds:.1f}s ({i+1}/{len(self.data.units)})', flush=True)
        return cube, selected

    def final_predictions(self, selection):
        train = np.arange(self.data.n_train_cases)
        test = np.arange(self.data.n_train_cases, self.data.n_all_cases)
        result = None
        for member, weight in selection['weights'].items():
            if weight <= 0:
                continue
            values, _ = self.run_stage(member, train, test, 'final', selection['rounds'][member])
            if result is None:
                result = np.zeros_like(values)
            result += float(weight) * values
        if result is None or abs(sum(selection['weights'].values()) - 1) > 1e-8:
            raise ValueError('Invalid ensemble weights')
        return np.clip(result, 0, 1)

## features — definitions only

In [6]:
"""Feature-only refinements for the frozen notebook 12/13 estimators."""
import numpy as np
import pandas as pd
import yaml


def price_competition(price, widths=(24, 72, 168)):
    """Strictly better alternative hours within this case, excluding current hour.

    Compute relative position and mean forgone price premium separately before
    and after the current hour. No edge replication or cross-case windows.
    """
    price = np.asarray(price, dtype=np.float64)
    n, horizon = price.shape
    ahead_count = np.zeros_like(price); past_count = np.zeros_like(price)
    ahead_gain = np.zeros_like(price); past_gain = np.zeros_like(price)
    scale = price.std(axis=1)[:, None] + 1.
    output = {}
    for distance in range(1, max(widths)):
        if distance < horizon:
            gap = price[:, distance:] - price[:, :-distance]
            ahead_count[:, :-distance] += gap > 0
            past_count[:, distance:] += gap < 0
            ahead_gain[:, :-distance] += np.maximum(gap, 0)
            past_gain[:, distance:] += np.maximum(-gap, 0)
        width = distance + 1
        if width in widths:
            ahead_n = np.maximum(1, np.minimum(width-1, horizon-1-np.arange(horizon)))
            past_n = np.maximum(1, np.minimum(width-1, np.arange(horizon)))
            output[f'v4_better_future_fraction_{width}'] = ahead_count / ahead_n
            output[f'v4_better_past_fraction_{width}'] = past_count / past_n
            output[f'v4_future_premium_{width}'] = ahead_gain / ahead_n / scale
            output[f'v4_past_premium_{width}'] = past_gain / past_n / scale
    return {k: v.astype(np.float32) for k, v in output.items()}


def build_refinement_features(data):
    n = data.n_all_cases
    price = data.X_all['price'].to_numpy().reshape(n, HORIZON)
    common = pd.DataFrame({k: v.ravel() for k, v in price_competition(price).items()})
    starts = pd.DatetimeIndex(pd.concat([data.train_cases, data.test_cases])['starttime'])
    times = starts.repeat(HORIZON) + pd.to_timedelta(np.tile(np.arange(HORIZON), n), unit='h')
    with open(data.root / 'data/extended/Tokke_Vinje_topology.yaml') as stream:
        topology = yaml.safe_load(stream)
    volumes = read_time_csv(data.root / 'data/kernel/Historical_volume_2015_2024.csv').set_index('date').reindex(starts)
    water = read_time_csv(data.root / 'data/kernel/Synthetic_water_value_2015_2024.csv').set_index('date').reindex(starts + pd.Timedelta(days=7))
    inflow = read_time_csv(data.root / 'data/kernel/Historical_inflow_1958_2025.csv').set_index('date').reindex(times)
    minimum = read_time_csv(data.root / 'data/extended/Constraint_min_volume.csv').set_index('date').reindex(times)
    price_only, combined, audit = {}, {}, []
    for plant in topology['model']['plant']:
        sources = plant_sources(topology, plant)
        cap = sum(topology['model']['reservoir'][s]['max_vol'] for s in sources)
        terminal = sum(water[s].to_numpy()[:, None] * topology['model']['reservoir'][s]['max_vol'] for s in sources) / cap
        margin = price - terminal
        flow = inflow[sources].sum(axis=1).to_numpy().reshape(n, HORIZON)
        initial = volumes[sources].sum(axis=1).to_numpy()[:, None]
        floor = sum((minimum[s].to_numpy().reshape(n, HORIZON) for s in sources if s in minimum), np.zeros_like(price))
        out = {}

        def add(name, values):
            values = np.broadcast_to(values, (n, HORIZON)).ravel().astype(np.float32)
            if not np.isfinite(values).all():
                raise ValueError(f'Non-finite feature {plant}/{name}')
            out['v4_' + name] = values

        prefix = np.arange(1, HORIZON+1)
        suffix = prefix[::-1]
        positive = margin > 0
        add('profitable_prefix_fraction', np.cumsum(positive, axis=1) / prefix)
        add('profitable_suffix_fraction', np.cumsum(positive[:, ::-1], axis=1)[:, ::-1] / suffix)
        inflow_cumulative = np.cumsum(flow, axis=1)
        flow_total = flow.sum(axis=1)[:, None]
        fraction_arrived = np.divide(inflow_cumulative, flow_total,
                                     out=np.zeros_like(inflow_cumulative), where=flow_total > 0)
        profit = np.maximum(margin, 0)
        profit_total = profit.sum(axis=1)[:, None]
        fraction_profit_elapsed = np.divide(np.cumsum(profit, axis=1), profit_total,
                                            out=np.zeros_like(profit), where=profit_total > 0)
        add('inflow_arrival_fraction', fraction_arrived)
        add('profit_elapsed_fraction', fraction_profit_elapsed)
        add('inflow_minus_profit_timing', fraction_arrived - fraction_profit_elapsed)
        add('no_profitable_hours', profit_total == 0)
        add('no_inflow', flow_total == 0)
        for width in (24, 72, 168):
            flow_window = window_sum(flow, width)
            weighted = np.divide(window_sum(flow*price, width), flow_window,
                                 out=np.zeros_like(price, dtype=np.float64), where=flow_window > 0)
            add(f'inflow_weighted_price_gap_{width}', np.where(flow_window > 0, weighted-price, 0))
            add(f'profitable_inflow_fill_{width}', window_sum(flow*positive, width)*.0036/cap)
        future_floor = np.maximum.accumulate(floor[:, ::-1], axis=1)[:, ::-1]
        add('upcoming_floor_increase_fill', (future_floor-floor)/cap)
        add('initial_slack_to_future_floor', (initial-future_floor)/cap)
        add('hours_until_highest_remaining_price', _hours_to_future_max(price))
        add('profitable_hours_remaining', np.cumsum(positive[:, ::-1], axis=1)[:, ::-1])
        water_features = pd.DataFrame(out)
        price_only[plant] = common
        combined[plant] = pd.concat([common, water_features], axis=1)
        audit.append(dict(plant=plant, sources='|'.join(sources),
                          price_features=len(common.columns), timing_features=len(water_features.columns)))
    return {'competition': price_only, 'competition_timing': combined}, pd.DataFrame(audit)


def _hours_to_future_max(price):
    n, horizon = price.shape
    result = np.zeros_like(price)
    best = np.full(n, -np.inf)
    index = np.full(n, horizon-1)
    for t in range(horizon-1, -1, -1):
        update = price[:, t] >= best
        best = np.maximum(best, price[:, t])
        index = np.where(update, t, index)
        result[:, t] = index-t
    return result


def screen_refinements(runner, fixed_rounds):
    """Controlled ablation: same per-unit rounds/settings for all feature variants.

    Choose the best new feature variant per backend on 2021, then audit the
    resulting fixed 50/50 blend on 2022. These years have been reused historically.
    """
    rows, dev_scores = [], {}
    train, valid = purged_year(runner.data, 2021)
    for member, spec in runner.specs.items():
        key = 'xgb_v2' if spec['backend'] == 'xgboost' else 'lgb_v2'
        p, _ = runner.run_stage(member, train, valid, 'ablation_2021', fixed_rounds[key])
        score = score_cube(runner.data.target_cube[valid], p)
        dev_scores[member] = score
        rows.append(dict(year=2021, model=member, micro_auc=score))
        print(f'2021 | {member}: {score:.6f}', flush=True)
    chosen = [max((m for m,s in runner.specs.items() if s['backend']==backend and s['features']!='physics'),
                  key=lambda m: dev_scores[m]) for backend in ('xgboost','lightgbm')]
    train, valid = purged_year(runner.data, 2022)
    results = {}
    for member in ['xgb_v2','lgb_v2'] + chosen:
        key = 'xgb_v2' if runner.specs[member]['backend']=='xgboost' else 'lgb_v2'
        results[member], _ = runner.run_stage(member, train, valid, 'audit_2022', fixed_rounds[key])
        rows.append(dict(year=2022, model=member, micro_auc=score_cube(runner.data.target_cube[valid],results[member])))
    new = (results[chosen[0]] + results[chosen[1]])*.5
    old = (results['xgb_v2']+results['lgb_v2'])*.5
    new_score, old_score = (score_cube(runner.data.target_cube[valid],p) for p in (new,old))
    rows += [dict(year=2022,model='new_blend',micro_auc=new_score),
             dict(year=2022,model='baseline_blend',micro_auc=old_score)]
    selection = dict(weights={m:.5 for m in chosen},
                     rounds={m:fixed_rounds['xgb_v2' if runner.specs[m]['backend']=='xgboost' else 'lgb_v2'] for m in chosen},
                     source_digest=runner.source_digest, raw_digest=runner.raw, specs=runner.specs,
                     audit_new_auc=new_score, audit_old_auc=old_score, passed_audit=bool(new_score>old_score))
    table = pd.DataFrame(rows)
    table.to_csv(runner.directory/'ablation_scores.csv',index=False)
    save_json(runner.directory/'selection.json',selection)
    print(f'2022 blend: new {new_score:.6f}, baseline {old_score:.6f}',flush=True)
    if not selection['passed_audit']:
        print('New features did not improve the historical audit. CSVs remain experimental.',flush=True)
    return selection,table

## Build feature groups — no training

In [7]:
data = build_project_data(ROOT)
extra, local, mapping = build_research_features(data)
additions, feature_summary = build_refinement_features(data)
runner = ChallengerRunner(data, extra, local, additions, SPECS, THREADS, SOURCE_DIGEST)
print('Cases:', data.n_train_cases, 'training;', data.n_test_cases, 'test')
display(feature_summary)
feature_summary.to_csv(runner.directory/'feature_groups.csv',index=False)

Cases: 2922 training; 725 test


,plant,sources,price_features,timing_features
0,Hogga,Bandak,12,17
1,Tokke,Vinjevatn,12,17
2,Lio,Byrtevatn,12,17
3,Byrte,Botnedalsvatn,12,17
4,Vinje,Vaamarvatn,12,17
5,Haukeli,Vatjern,12,17
6,Songa,Bitdalsvatn|Songavatn,12,17
7,Kjela,Foersvatn,12,17
8,Vesle Kjela,Kjelavatn,12,17


## TRAINING A — controlled feature comparison

In [8]:
# TRAINING A: fixed-parameter feature ablation, then historical audit.
selection, scores = screen_refinements(runner, BASELINE_ROUNDS)
display(scores.pivot(index='model', columns='year', values='micro_auc'))
print('Chosen new-model pair:', selection['weights'])
print('Improved the 2022 audit:', selection['passed_audit'])

ablation_2021 | xgb_v2 | Hogga_G1 | up to 410 rounds
  trained: 410 rounds, 10.8s (1/14)
ablation_2021 | xgb_v2 | Tokke_G1 | up to 450 rounds
  trained: 450 rounds, 13.1s (2/14)
ablation_2021 | xgb_v2 | Tokke_G2 | up to 854 rounds
  trained: 854 rounds, 21.7s (3/14)
ablation_2021 | xgb_v2 | Tokke_G3 | up to 587 rounds
  trained: 587 rounds, 16.5s (4/14)
ablation_2021 | xgb_v2 | Tokke_G4 | up to 693 rounds
  trained: 693 rounds, 18.3s (5/14)
ablation_2021 | xgb_v2 | Lio_G1 | up to 249 rounds
  trained: 249 rounds, 6.9s (6/14)
ablation_2021 | xgb_v2 | Byrte_G1 | up to 292 rounds
  trained: 292 rounds, 7.9s (7/14)
ablation_2021 | xgb_v2 | Vinje_G1 | up to 658 rounds
  trained: 658 rounds, 18.5s (8/14)
ablation_2021 | xgb_v2 | Vinje_G2 | up to 404 rounds
  trained: 404 rounds, 12.0s (9/14)
ablation_2021 | xgb_v2 | Vinje_G3 | up to 607 rounds
  trained: 607 rounds, 16.0s (10/14)
ablation_2021 | xgb_v2 | Haukeli_G1 | up to 223 rounds
  trained: 223 rounds, 7.0s (11/14)
ablation_2021 | xgb_v2

year,2021,2022
model,,
baseline_blend,NaN,0.986219
lgb_competition,0.983922,0.985954
lgb_competition_timing,0.983807,NaN
lgb_v2,0.983760,0.986065
new_blend,NaN,0.986236
xgb_competition,0.984193,0.985677
xgb_competition_timing,0.984088,NaN
xgb_v2,0.984074,0.985523


Chosen new-model pair: {'xgb_competition': 0.5, 'lgb_competition': 0.5}
Improved the 2022 audit: True


## TRAINING B — new models and submission CSVs

In [9]:
# TRAINING B: exactly two selected new models per generator, trained on all history.
# Restart: run setup/definition/build cells, then this cell to resume final fitting.
selection = json.loads((runner.directory/'selection.json').read_text())
if (selection['source_digest'] != SOURCE_DIGEST or selection['raw_digest'] != runner.raw
        or selection['specs'] != SPECS):
    raise ValueError('Saved selection is for a different configuration. Rerun TRAINING A.')
train = np.arange(data.n_train_cases)
test = np.arange(data.n_train_cases, data.n_all_cases)
blend = np.zeros((data.n_test_cases,len(data.units),HORIZON),np.float32)
for member, weight in selection['weights'].items():
    predictions, _ = runner.run_stage(member, train, test, 'final', selection['rounds'][member])
    family = 'xgb' if SPECS[member]['backend']=='xgboost' else 'lgbm'
    path, _ = make_submission(data, predictions, f'{family}_physics_features_v4.csv')
    print('Single-model CSV:',path)
    blend += weight * predictions
submission_path, submission = make_submission(data,blend,'physics_features_v4_blend.csv')
print('NEW 50/50 BLEND READY:',submission_path)
print('Improved historical audit:',selection['passed_audit'])
if not selection['passed_audit']:
    print('This is still the NEW-model blend, not a baseline fallback. Historical results do not support an improvement.')
print('Total minutes:',round((time.perf_counter()-NOTEBOOK_STARTED)/60,1))
display(submission.iloc[:3,:8])

final | xgb_competition | Hogga_G1 | up to 410 rounds
  trained: 410 rounds, 15.1s (1/14)
final | xgb_competition | Tokke_G1 | up to 450 rounds
  trained: 450 rounds, 19.8s (2/14)
final | xgb_competition | Tokke_G2 | up to 854 rounds
  trained: 854 rounds, 31.7s (3/14)
final | xgb_competition | Tokke_G3 | up to 587 rounds
  trained: 587 rounds, 24.1s (4/14)
final | xgb_competition | Tokke_G4 | up to 693 rounds
  trained: 693 rounds, 27.0s (5/14)
final | xgb_competition | Lio_G1 | up to 249 rounds
  trained: 249 rounds, 9.9s (6/14)
final | xgb_competition | Byrte_G1 | up to 292 rounds
  trained: 292 rounds, 10.9s (7/14)
final | xgb_competition | Vinje_G1 | up to 658 rounds
  trained: 658 rounds, 25.7s (8/14)
final | xgb_competition | Vinje_G2 | up to 404 rounds
  trained: 404 rounds, 16.7s (9/14)
final | xgb_competition | Vinje_G3 | up to 607 rounds
  trained: 607 rounds, 22.9s (10/14)
final | xgb_competition | Haukeli_G1 | up to 223 rounds
  trained: 223 rounds, 10.0s (11/14)
final | x

,Run No,result_committed_Hogga_G1_t0,result_committed_Hogga_G1_t1,result_committed_Hogga_G1_t2,result_committed_Hogga_G1_t3,result_committed_Hogga_G1_t4,result_committed_Hogga_G1_t5,result_committed_Hogga_G1_t6
0,2923,0.142643,0.082881,0.049999,0.031725,0.022815,0.019646,0.009365
1,2924,0.844539,0.843768,0.852017,0.861462,0.844429,0.881003,0.910118
2,2925,0.999432,0.999431,0.999426,0.998978,0.999452,0.999478,0.999531


## Export a self-contained short notebook

In [10]:
# Freeze the selected pair into a self-contained final-only reproduction notebook.
original = ROOT/'notebooks/17_physics_feature_refinement.ipynb'
document = json.loads(original.read_text(encoding='utf-8'))
cells = []
for c in document['cells']:
    if c.get('id') in ('resources','config','pipeline','physics','runner','features','build'):
        c = json.loads(json.dumps(c))
        c['execution_count'],c['outputs'] = None,[]
        if c['id']=='resources':
            c['source'] = ''.join(c['source']).replace("RUN_PROFILE = 'local'", "RUN_PROFILE = 'professor'").splitlines(keepends=True)
        if c['id']=='features':
            c['source'] = ''.join(c['source']).split('def screen_refinements(')[0].splitlines(keepends=True)
        cells.append(c)
frozen = 'selection = '+repr(selection)+'\n'
fitting = ('prediction = runner.final_predictions(selection)\n'
           'path, submission = make_submission(data, prediction, "physics_features_v4_blend.csv")\n'
           'print("Saved:",path)\n'
           'print("Elapsed minutes:",(time.perf_counter()-NOTEBOOK_STARTED)/60)\n')
for ident,source in [('frozen',frozen),('final',fitting)]:
    cells.append(dict(cell_type='code',id=ident,metadata={},execution_count=None,outputs=[],source=source.splitlines(keepends=True)))
intro = ('# Short notebook — physics feature refinement blend\n\n'
         '**Henrik Remmen (541747), Group 119, [119] SINTEF Databreach**\n\n'
         'Offline raw-data reproduction with four CPU threads. Models and rounds match '
         'notebooks 12/13; feature variants selected on 2021, audited on 2022 in notebook 17. '
         'Run all cells. No pretrained artifacts or other notebooks are required.\n')
cells.insert(0,dict(cell_type='markdown',id='intro',metadata={},source=intro.splitlines(keepends=True)))
document['cells']=cells
short_path = runner.directory/'Short_notebook_physics_features_v4.ipynb'
short_path.write_text(json.dumps(document,indent=1,ensure_ascii=False)+'\n',encoding='utf-8')
print('Final-only notebook:',short_path)

Final-only notebook: /Users/henrik/Documents/H26/Moderne Maksinlæring/sintef_project/artifacts/physics_features_v4/Short_notebook_physics_features_v4.ipynb


## Outputs and research rationale

Upload candidate: `submissions/physics_features_v4_blend.csv`.
Individual candidates: `submissions/xgb_physics_features_v4.csv` and `submissions/lgbm_physics_features_v4.csv`.
Diagnostics and short notebook: `artifacts/physics_features_v4/`.

Why these hypotheses: the existing features encode weekly rank, mean prices and price-minus-water
value, but do not directly describe competing higher-price hours separately before and after
the current hour. Existing cumulative inflow also lacks explicit alignment with profitable hours.
The new variables quantify these relationships without fitting another model or pretending
to reproduce the full hydraulic solver. They may help or hurt; the ablation measures this.

[SINTEF SHOP water values](https://docs.shop.sintef.energy/examples/water_values/water_values.html)
explain the opportunity cost of stored water and the interpretation of local EUR/MWh values.
[Scikit-learn temporal validation](https://scikit-learn.org/stable/modules/cross_validation.html#time-series-split)
supports testing on later observations; here 168-hour case overlaps are additionally purged.
These sources motivate the design, not a claimed improvement. Public scores never enter selection.